In [1]:
# ============================================================
# INTRUSION DETECTION - BINARY SUBMISSION PIPELINE
# Output format:
# id,prediction
# 1,0
# 2,1
# where 0 = normal, 1 = attack
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import os
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

# Optional LightGBM
try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False


# ============================================================
# 1. LOAD TRAIN DATA
# ============================================================
LOCAL_DATA_DIR = Path(os.environ.get("LOCAL_DATA_DIR", Path.cwd())).parent.resolve()
DATA_DIR = LOCAL_DATA_DIR / "data"
SUBMISSION_DIR = LOCAL_DATA_DIR / "submission"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

print(LOCAL_DATA_DIR)

# Load data
test_df = pd.read_csv(DATA_DIR / "predict.csv")
train_df = pd.read_csv(DATA_DIR / "training.csv")


print("Training data shape:", train_df.shape)
print("Prediction data shape:", test_df.shape)
display(train_df.head())


#pd.set_option("display.max_columns", 200)
#pd.set_option("display.width", 140)
#train_path = "output_CS.csv"     # your uploaded training file
#test_path = "test.csv"           # replace with actual test file name when available

#train_df = pd.read_csv(train_path)

#print("Train shape:", train_df.shape)
#print("Train columns:", train_df.columns.tolist())


# ============================================================
# 2. CREATE BINARY TARGET
# ============================================================
# attack_cat is used to create:
# 0 = normal
# 1 = attack
train_df.columns = train_df.columns.str.strip()

if "attack_cat" not in train_df.columns:
    raise ValueError("attack_cat column not found in training data.")

train_df["prediction"] = (train_df["attack_cat"].astype(str).str.lower() != "normal").astype(int)

# Drop leakage columns from features
drop_cols = [col for col in ["attack_cat", "label", "prediction"] if col in train_df.columns]
X = train_df.drop(columns=drop_cols)
y = train_df["prediction"]

print("\nBinary target distribution:")
print(y.value_counts())


# ============================================================
# 3. OPTIONAL FEATURE ENGINEERING
# ============================================================
def add_ratio_features(df):
    df = df.copy()
    eps = 1e-6

    if {"sbytes", "dbytes"}.issubset(df.columns):
        df["bytes_ratio_sd"] = df["sbytes"] / (df["dbytes"] + eps)
        df["bytes_total"] = df["sbytes"] + df["dbytes"]

    if {"spkts", "dpkts"}.issubset(df.columns):
        df["pkts_ratio_sd"] = df["spkts"] / (df["dpkts"] + eps)
        df["pkts_total"] = df["spkts"] + df["dpkts"]

    if {"sload", "dload"}.issubset(df.columns):
        df["load_ratio_sd"] = df["sload"] / (df["dload"] + eps)

    if {"sinpkt", "dinpkt"}.issubset(df.columns):
        df["inpkt_ratio_sd"] = df["sinpkt"] / (df["dinpkt"] + eps)

    return df


# ============================================================
# 4. IDENTIFY COLUMN TYPES
# ============================================================
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = [c for c in X.columns if c not in categorical_cols]

log_candidate_cols = [
    c for c in numeric_cols
    if c in [
        "dur", "spkts", "dpkts", "sbytes", "dbytes", "rate", "sload", "dload",
        "sinpkt", "dinpkt", "sjit", "djit", "tcprtt", "synack", "ackdat",
        "ct_srv_src", "ct_dst_ltm", "ct_src_dport_ltm", "ct_dst_sport_ltm",
        "ct_dst_src_ltm"
    ]
]

def log_transform_selected(df_num):
    # works with DataFrame (pre-fit) and numpy array (after imputer)
    if isinstance(df_num, pd.DataFrame):
        df_out = df_num.copy()
        for col in log_candidate_cols:
            if col in df_out.columns:
                df_out[col] = np.log1p(np.clip(df_out[col], a_min=0, a_max=None))
        return df_out

    if isinstance(df_num, np.ndarray):
        df_out = df_num.copy()
        for col in log_candidate_cols:
            if col in numeric_cols:
                idx = numeric_cols.index(col)
                df_out[:, idx] = np.log1p(np.clip(df_out[:, idx], a_min=0, a_max=None))
        return df_out

    # fallback for other input types
    df_out = np.array(df_num, copy=True)
    for col in log_candidate_cols:
        if col in numeric_cols:
            idx = numeric_cols.index(col)
            df_out[:, idx] = np.log1p(np.clip(df_out[:, idx], a_min=0, a_max=None))
    return df_out


# ============================================================
# 5. PREPROCESSOR
# ============================================================
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("log_transform", FunctionTransformer(log_transform_selected, validate=False)),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)


# ============================================================
# 6. TRAIN / VALIDATION SPLIT
# ============================================================
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# ============================================================
# 7. MODEL CANDIDATES
# ============================================================
models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced_subsample",
        n_jobs=-1
    ),
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    )
}

if HAS_LGBM:
    models["LightGBM"] = LGBMClassifier(
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight="balanced",
        random_state=42
    )


# ============================================================
# 8. TRAIN AND PICK BEST MODEL
# ============================================================
best_model_name = None
best_pipeline = None
best_f1 = -1

for model_name, model in models.items():
    pipeline = Pipeline(steps=[
        ("feature_eng", FunctionTransformer(add_ratio_features, validate=False)),
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_valid)

    #model.fit(X_train, y_train)
    #preds = model.predict(X_valid)

    acc = accuracy_score(y_valid, preds)
    bal_acc = balanced_accuracy_score(y_valid, preds)
    prec = precision_score(y_valid, preds, zero_division=0)
    rec = recall_score(y_valid, preds, zero_division=0)
    f1 = f1_score(y_valid, preds, zero_division=0)

    print(f"\n===== {model_name} =====")
    print(f"Accuracy:          {acc:.4f}")
    print(f"Balanced Accuracy: {bal_acc:.4f}")
    print(f"Precision:         {prec:.4f}")
    print(f"Recall:            {rec:.4f}")
    print(f"F1 Score:          {f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_valid, preds, target_names=["Normal", "Attack"], zero_division=0))
    print("Confusion Matrix:")
    print(confusion_matrix(y_valid, preds))

    if f1 > best_f1:
        best_f1 = f1
        best_model_name = model_name
        best_pipeline = pipeline

print(f"\nBest model selected: {best_model_name} with F1 = {best_f1:.4f}")


# ============================================================
# 9. RETRAIN BEST MODEL ON FULL TRAINING DATA
# ============================================================
best_pipeline.fit(X, y)

joblib.dump(best_pipeline, "best_binary_attack_model.pkl")
print("Saved model: best_binary_attack_model.pkl")


# ============================================================
# 10. LOAD TEST DATA
# ============================================================
# The test file should contain the same feature columns as training,
# except no attack_cat / label / prediction target columns.

#test_df = pd.read_csv(test_path)
#test_df.columns = test_df.columns.str.strip()

#print("\nTest shape:", test_df.shape)
#print("Test columns:", test_df.columns.tolist())


# ============================================================
# 11. BUILD SUBMISSION IDS
# ============================================================
# Required format:
# id,prediction
# 1,0
# 2,1

# If the competition test set already has an 'id' column, use it.
# Otherwise generate sequential ids starting from 1.
if "id" in test_df.columns:
    submission_id = test_df["id"]
    X_test = test_df.drop(columns=["id"])
else:
    submission_id = pd.Series(np.arange(1, len(test_df) + 1), name="id")
    X_test = test_df.copy()

# Drop any accidental target columns if present
for col in ["attack_cat", "label", "prediction"]:
    if col in X_test.columns:
        X_test = X_test.drop(columns=[col])

# Predict:
# 0 = normal
# 1 = attack
test_preds = best_pipeline.predict(X_test)

submission = pd.DataFrame({
    "id": submission_id,
    "prediction": test_preds.astype(int)
})

# Ensure exact column order
submission = submission[["id", "prediction"]]

# Save submission file
submission.to_csv("submission.csv", index=False)

print("\nSubmission file created: submission.csv")
print(submission.head(10))


# ============================================================
# 12. OPTIONAL FORMAT CHECK
# ============================================================
# Basic checks to ensure the submission is valid
assert list(submission.columns) == ["id", "prediction"], "Submission columns must be exactly: id,prediction"
assert submission["prediction"].isin([0, 1]).all(), "Prediction column must contain only 0 or 1"

print("\nSubmission format check passed.")

/home/user/NTU_M3_Kaggle_MoneyBall
Training data shape: (175341, 36)
Prediction data shape: (82332, 36)


,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sload,...,trans_depth,response_body_len,ct_src_dport_ltm,ct_dst_sport_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,is_sm_ips_ports,attack_cat,label
0,0.121478,tcp,-,FIN,6,4,258,172,74.087490,14158.9420,...,0,0,1,1,0,0,0,0,Normal,0
1,0.649902,tcp,-,FIN,14,38,734,42014,78.473370,8395.1120,...,0,0,1,1,0,0,0,0,Normal,0
2,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,1572.2719,...,0,0,1,1,0,0,0,0,Normal,0
3,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,2740.1790,...,0,0,1,1,1,1,0,0,Normal,0
4,0.449454,tcp,-,FIN,10,6,534,268,33.373825,8561.4990,...,0,0,2,1,0,0,0,0,Normal,0



Binary target distribution:
prediction
1    119341
0     56000
Name: count, dtype: int64

===== RandomForest =====
Accuracy:          0.9500
Balanced Accuracy: 0.9379
Precision:         0.9559
Recall:            0.9713
F1 Score:          0.9635

Classification Report:
              precision    recall  f1-score   support

      Normal       0.94      0.90      0.92     11200
      Attack       0.96      0.97      0.96     23869

    accuracy                           0.95     35069
   macro avg       0.95      0.94      0.94     35069
weighted avg       0.95      0.95      0.95     35069

Confusion Matrix:
[[10130  1070]
 [  685 23184]]

===== ExtraTrees =====
Accuracy:          0.9478
Balanced Accuracy: 0.9355
Precision:         0.9545
Recall:            0.9695
F1 Score:          0.9619

Classification Report:
              precision    recall  f1-score   support

      Normal       0.93      0.90      0.92     11200
      Attack       0.95      0.97      0.96     23869

    accuracy